# 053 — Transfer Learning

Keep the part that generalises, replace the part that does not. This notebook
reproduces the lesson's **honest** result: a measured case where transfer
learning makes things **worse**, because the precondition matters more than the
recipe.

Needs `scikit-learn` (the Olivetti faces part downloads ~5 MB once).
**Parts B and C train 8 models per row and take several minutes** - that is
the price of averaging over 8 random splits rather than trusting one.

| Part | What we check |
|---|---|
| A | what you actually train: 12.5% of the parameters |
| B | transfer against scratch on digits - transfer **loses** |
| C | the same on faces - transfer loses again |
| D | why: a frozen dense layer is not a feature extractor |

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Part A — What feature extraction leaves trainable

In [ ]:
CONV_BASE = 14_714_688           # VGG16, include_top=False
flat = 4 * 4 * 512               # a 150x150 input after five poolings
head = flat * 256 + 256 + 256 + 1
total = CONV_BASE + head
print(f"frozen base   {CONV_BASE:>12,}")
print(f"new head      {head:>12,}")
print(f"total         {total:>12,}")
print(f"trainable     {head / total:>12.1%}")
assert round(head / total * 100, 1) == 12.5
print()
print("Fine-tuning block 5 as well unfreezes 7,079,424 more:")
block5 = 3 * ((3 * 3 * 512 + 1) * 512)
print(f"  block 5 convolutions {block5:,} -> trainable "
      f"{(head + block5) / total:.1%}")

## Part B — Transfer learning losing, on digits

Pretrain on digits 0-4, freeze the hidden layer, reuse it as a "feature
extractor" for digits 5-9. Compare against the same architecture trained from
scratch on the same small data. Eight random splits each.

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

def experiment(X, y, split_at, sizes, hidden, label, seeds=8):
    source, target = y < split_at, y >= split_at
    base = MLPClassifier(hidden_layer_sizes=(hidden,), max_iter=1500, random_state=0)
    base.fit(X[source], y[source])

    def frozen(Z):                           # the reused layer, ReLU and all
        return np.maximum(0, Z @ base.coefs_[0] + base.intercepts_[0])

    Xt, yt = X[target], y[target]
    print(f"{label}  (pretrained on {source.sum()} images)")
    print(f"  {'n train':>8} {'scratch':>9} {'transfer':>9} {'gain':>7}")
    out = []
    for n in sizes:
        scratch, transfer = [], []
        for seed in range(seeds):
            Xa, Xb, ya, yb = train_test_split(
                Xt, yt, train_size=n, random_state=seed, stratify=yt)
            scratch.append(MLPClassifier(hidden_layer_sizes=(hidden,), max_iter=1500,
                                         random_state=0).fit(Xa, ya).score(Xb, yb))
            transfer.append(MLPClassifier(hidden_layer_sizes=(), max_iter=1500,
                                          random_state=0)
                            .fit(frozen(Xa), ya).score(frozen(Xb), yb))
        s, t = float(np.mean(scratch)), float(np.mean(transfer))
        out.append((n, s, t))
        print(f"  {n:>8} {s:>8.1%} {t:>8.1%} {t - s:>+6.1%}")
    return out

digits = load_digits()
rows = experiment(digits.data / 16.0, digits.target, 5, (25, 50, 100, 200), 64,
                  "sklearn digits: pretrain on 0-4, transfer to 5-9")
assert all(t < s for _, s, t in rows)        # transfer loses at EVERY size
print("\ntransfer lost at every training-set size")

## Part C — The same experiment on faces

In [ ]:
try:
    from sklearn.datasets import fetch_olivetti_faces
    faces = fetch_olivetti_faces(shuffle=False)
    face_rows = experiment(faces.data, faces.target, 20, (40, 60, 100), 128,
                           "Olivetti faces: 20 people -> 20 others")
    assert all(t < s for _, s, t in face_rows)
    print("\nlost again, and by more: a harder task does not rescue it")
except Exception as exc:
    print(f"skipped - dataset unavailable offline ({type(exc).__name__})")

## Part D — Why it lost, and when it wins

The reused layer was trained to separate the **source classes**. Its outputs
are source-class evidence, not general structure. A dense layer on raw pixels
has no locality and no weight sharing, so nothing it learns transfers the way a
convolutional stack's early layers do.

In [ ]:
# the reported behaviour when the reused part IS a convolutional stack trained
# on 14 million natural images - these are documented figures for VGG16 on
# dogs-vs-cats, NOT measured here (TensorFlow is not installed)
reported = [("hand-built CNN, lesson 049", 81),
            ("frozen VGG16 base + new head", 90),
            ("... plus augmentation", 91),
            ("... fine-tuning block 5, low LR", 95)]
for label, acc in reported:
    print(f"  {label:<34} ~{acc}%   (reported)")
print()
print("The precondition: the reused layers must encode structure true of the")
print("DOMAIN, not of the source labels. Conv layers on natural images do.")
print("A 128-unit dense bottleneck does not - which is what Parts B and C")
print("measured.")

## What this notebook established

- Feature extraction on a VGG16 base leaves **12.5%** of the parameters
  trainable; unfreezing block 5 raises that by 7,079,424 more.
- Reusing a **frozen dense hidden layer** lost to training from scratch at
  **every** training-set size, on both digits and faces, averaged over 8 splits.
- The reason is the precondition, not the recipe: a dense bottleneck encodes
  source-label evidence, while convolutional layers on natural images encode
  domain structure.
- The ~81% → ~95% dogs-versus-cats progression is **reported**, not measured
  here.

## Exercises

1. Unfreeze the reused layer (set `hidden_layer_sizes=(64,)` and warm-start from
   `base.coefs_`). Does transfer catch up?
2. Make source and target closer — pretrain on 0-7, transfer to 8-9. Does the
   loss shrink?
3. Replace the dense bottleneck with a hand-written 3x3 convolution bank and
   repeat. Which direction does the gain move, and does that match the
   explanation above?